# Agent 2 — Teams Alert Dispatcher

Sends Teams MessageCard alerts for CRITICAL and WARNING
reconciliation runs that have not yet been alerted.

Reads from:
  uc2_reconciliation_run_b
  uc2_reconciliation_results_b

Updates:
  uc2_reconciliation_run_b (alert_sent, alert_sent_timestamp)

## Cell 1: Imports and Configuration

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

RECONCILIATION_RUN     = f"{CATALOG}.{SCHEMA}.uc2_reconciliation_run_b"
RECONCILIATION_RESULTS = f"{CATALOG}.{SCHEMA}.uc2_reconciliation_results_b"

# ── Teams Webhook ──────────────────────────────────────────────────────────

TEAMS_WEBHOOK_URL = dbutils.secrets.get(
    scope="agent1-secrets",
    key  ="teams-webhook-url"
)

ALERT_CHANNEL     = "platform-support"
ALERT_ON_STATUSES = {"CRITICAL", "WARNING"}

STATUS_EMOJI = {
    "CRITICAL": "🔴",
    "WARNING" : "🟡",
    "PASS"    : "🟢",
    "ERROR"   : "⚫"
}

THEME_COLOR = {
    "CRITICAL": "FF0000",
    "WARNING" : "FFA500",
    "PASS"    : "00FF00"
}

print("=" * 60)
print("AGENT 2 — TEAMS ALERT DISPATCHER")
print("=" * 60)
print(f"Alert channel  : {ALERT_CHANNEL}")
print(f"Webhook loaded : {'YES' if TEAMS_WEBHOOK_URL else 'NO'}")
print(f"Started at     : {datetime.now()}")
print("=" * 60)

## Cell 2: Load Runs to Alert

In [0]:
def load_runs_to_alert() -> list:
    """
    Loads reconciliation runs that need alerts.
    Conditions:
      1. Status is CRITICAL or WARNING
      2. AI enrichment is COMPLETED or NOT_REQUIRED
      3. Alert has not been sent yet
    """
    df = (
        spark.table(RECONCILIATION_RUN)
             .filter(
                 F.col("overall_status").isin(list(ALERT_ON_STATUSES)) &
                 F.col("ai_enrichment_status").isin(
                     ["COMPLETED", "NOT_REQUIRED"]
                 ) &
                 (
                     F.col("alert_sent").isNull() |
                     (F.col("alert_sent") == False)
                 )
             )
             .orderBy("business_date", "source_fully_qualified_name")
    )

    runs = [row.asDict() for row in df.collect()]

    print(f"Runs requiring alerts: {len(runs)}")
    for r in runs:
        emoji = STATUS_EMOJI.get(r["overall_status"], "⚪")
        print(
            f"  {emoji} {r['business_date']}  "
            f"{r['source_fully_qualified_name'].split('.')[-1]:<35}  "
            f"Status: {r['overall_status']:<10}  "
            f"AI: {r['ai_enrichment_status']}"
        )

    return runs


runs_to_alert = load_runs_to_alert()

if not runs_to_alert:
    print("\nNo runs require alerts at this time.")
    NO_RUNS_TO_ALERT = True
else:
    NO_RUNS_TO_ALERT = False
    print(f"\nProceeding to dispatch {len(runs_to_alert)} alert(s).")

## Cell 3: Load Failed Checks

In [0]:
def load_failed_checks(reconciliation_run_id: str) -> list:
    """
    Loads WARNING and CRITICAL checks for a given run.
    Ordered by severity then check type.
    """
    df = (
        spark.table(RECONCILIATION_RESULTS)
             .filter(
                 (F.col("reconciliation_run_id") == reconciliation_run_id) &
                 (F.col("check_status").isin(["WARNING", "CRITICAL"]))
             )
             .orderBy(
                 F.when(F.col("check_status") == "CRITICAL", 1)
                  .otherwise(2),
                 "from_layer",
                 "check_type"
             )
    )
    return [row.asDict() for row in df.collect()]

print("Load failed checks function ready.")

## Cell 4: Build MessageCard

In [0]:
def format_check_line(check: dict) -> str:
    """
    Formats a single failed check as a readable line
    for the Teams card body.
    """
    emoji       = STATUS_EMOJI.get(check["check_status"], "⚪")
    check_type  = check["check_type"]
    from_layer  = check["from_layer"]
    to_layer    = check["to_layer"]
    metric      = check["metric_name"]
    source_val  = check["source_value"]
    target_val  = check["target_value"]
    deviation   = check["deviation_percentage"]

    if check_type in (
        "RAW_TO_SILVER_ROW_COUNT",
        "SILVER_TO_GOLD_ROW_COUNT"
    ):
        detail = (
            f"{from_layer}: {int(source_val):,} rows  →  "
            f"{to_layer}: {int(target_val):,} rows  "
            f"({deviation:.1f}% deviation)"
        )
    elif check_type == "LAYER_FRESHNESS":
        if target_val == 0:
            detail = f"{to_layer} has NO records for this date"
        else:
            detail = f"{to_layer} data present ({int(target_val):,} records)"
    else:
        detail = (
            f"{from_layer}: {source_val:,.2f}  →  "
            f"{to_layer}: {target_val:,.2f}  "
            f"({deviation:.2f}% deviation)"
        )

    return (
        f"{emoji} **{from_layer} → {to_layer}**  "
        f"{check_type}  \n"
        f"  {metric}: {detail}"
    )


def build_agent2_message_card(
    run_record    : dict,
    failed_checks : list
) -> dict:
    """
    Builds a Teams MessageCard for an Agent 2 alert.

    The card highlights which layer transition failed
    (Raw→Silver or Silver→Gold) which is the key
    differentiator from Agent 1 alerts.
    """
    fqn          = run_record["source_fully_qualified_name"]
    table_name   = fqn.split(".")[-1]
    biz_date     = str(run_record["business_date"])
    status       = run_record["overall_status"]
    run_id       = run_record["reconciliation_run_id"]
    total_run    = run_record["total_checks_run"]
    total_passed = run_record["total_checks_passed"]
    total_warned = run_record["total_checks_warned"]
    total_crit   = run_record["total_checks_critical"]
    agent1_run   = run_record.get("agent1_validation_run_id")

    emoji        = STATUS_EMOJI.get(status, "⚪")
    color        = THEME_COLOR.get(status, "0078D4")
    detected_at  = datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

    # Extract AI enrichment from first enriched failed check
    ai_summary    = None
    ai_action     = None
    ai_category   = None
    ai_confidence = None

    for check in failed_checks:
        if check.get("ai_summary"):
            ai_summary    = check["ai_summary"]
            ai_action     = check["ai_recommended_action"]
            ai_category   = check["ai_failure_category"]
            ai_confidence = check["ai_confidence_score"]
            break

    sections = []

    # ── Section 1: Header ──────────────────────────────────────────────────
    sections.append({
        "activityTitle"   : (
            f"{emoji} {status} — Cross-Layer Reconciliation Alert"
        ),
        "activitySubtitle": (
            "Agent 2 — Data Layer Reconciliation"
        ),
        "facts": [
            {
                "name" : "Source Table",
                "value": table_name
            },
            {
                "name" : "Full Name",
                "value": fqn
            },
            {
                "name" : "Business Date",
                "value": biz_date
            },
            {
                "name" : "Detected At",
                "value": detected_at
            },
            {
                "name" : "Overall Status",
                "value": f"{emoji} {status}"
            }
        ],
        "markdown": True
    })

    # ── Section 2: Check Summary ───────────────────────────────────────────
    sections.append({
        "title": "📊 Reconciliation Summary",
        "facts": [
            {
                "name" : "Total Checks Run",
                "value": str(total_run)
            },
            {
                "name" : "✅ Passed",
                "value": str(total_passed)
            },
            {
                "name" : "🟡 Warning",
                "value": str(total_warned)
            },
            {
                "name" : "🔴 Critical",
                "value": str(total_crit)
            },
            {
                "name" : "Agent 1 Context",
                "value": "Available ✅" if agent1_run else "Not Available"
            }
        ],
        "markdown": True
    })

    # ── Section 3: Failed Checks ───────────────────────────────────────────
    if failed_checks:
        failed_lines = [
            format_check_line(check)
            for check in failed_checks
        ]

        sections.append({
            "title"   : f"❌ Failed Checks ({len(failed_checks)})",
            "text"    : "\n\n".join(failed_lines),
            "markdown": True
        })

    # ── Section 4: AI Analysis ─────────────────────────────────────────────
    if ai_summary:
        sections.append({
            "title"   : "🤖 AI Cross-Layer Analysis",
            "text"    : ai_summary,
            "markdown": True
        })

        if ai_action:
            sections.append({
                "title"   : "✅ Recommended Action",
                "text"    : ai_action,
                "markdown": True
            })

        sections.append({
            "title": "AI Enrichment Details",
            "facts": [
                {
                    "name" : "Failure Category",
                    "value": ai_category or "N/A"
                },
                {
                    "name" : "AI Confidence",
                    "value": (
                        f"{ai_confidence:.0%}"
                        if ai_confidence else "N/A"
                    )
                }
            ],
            "markdown": True
        })

    # ── Section 5: Footer ──────────────────────────────────────────────────
    sections.append({
        "title": "🔍 Audit",
        "facts": [
            {
                "name" : "Reconciliation Run ID",
                "value": run_id
            },
            {
                "name" : "Agent 1 Run ID",
                "value": agent1_run or "N/A"
            },
            {
                "name" : "Catalog.Schema",
                "value": f"{CATALOG}.{SCHEMA}"
            }
        ],
        "markdown": True
    })

    return {
        "@type"     : "MessageCard",
        "@context"  : "http://schema.org/extensions",
        "themeColor": color,
        "summary"   : (
            f"Agent 2 Alert: {table_name} {status} on {biz_date}"
        ),
        "title"     : (
            f"{emoji} {status}: {table_name} — {biz_date}"
        ),
        "sections"  : sections
    }

print("MessageCard builder ready.")

## Cell 5: Send and Mark

In [0]:
def send_teams_alert(card: dict, run_id: str) -> bool:
    """
    Posts the card to Teams webhook.
    Returns True on success.
    """
    try:
        response = requests.post(
            TEAMS_WEBHOOK_URL,
            headers={"Content-Type": "application/json"},
            data=json.dumps(card),
            timeout=30
        )
        if response.status_code in (200, 202):
            logger.info(f"Alert sent for run {run_id}")
            return True
        else:
            logger.error(
                f"Alert failed. "
                f"Status: {response.status_code}. "
                f"Response: {response.text[:200]}"
            )
            return False
    except Exception as e:
        logger.error(f"Alert exception: {str(e)}")
        return False


def mark_alert_sent(run_id: str, success: bool):
    """
    Updates the run record to prevent duplicate alerts.
    """
    channel = ALERT_CHANNEL if success else "FAILED"
    spark.sql(f"""
        UPDATE {RECONCILIATION_RUN}
        SET
            alert_sent              = {str(success).lower()},
            alert_sent_timestamp    = '{datetime.now()}',
            alert_channel           = '{channel}'
        WHERE reconciliation_run_id = '{run_id}'
    """)

print("Send and mark functions ready.")

## Cell 6: Main Dispatch Loop

In [0]:
alerts_sent   = 0
alerts_failed = 0

if NO_RUNS_TO_ALERT:
    print("Skipping dispatch — no runs require alerts.")
else:
    print(f"Dispatching {len(runs_to_alert)} alert(s)...\n")

    for run_record in runs_to_alert:

        run_id     = run_record["reconciliation_run_id"]
        fqn        = run_record["source_fully_qualified_name"]
        table_name = fqn.split(".")[-1]
        biz_date   = run_record["business_date"]
        status     = run_record["overall_status"]
        emoji      = STATUS_EMOJI.get(status, "⚪")

        print(f"{'='*60}")
        print(f"{emoji} {table_name}  |  {biz_date}  |  {status}")
        print(f"{'='*60}")

        failed_checks = load_failed_checks(run_id)
        print(f"Failed checks : {len(failed_checks)}")

        try:
            card    = build_agent2_message_card(run_record, failed_checks)
            success = send_teams_alert(card, run_id)
        except Exception as e:
            print(f"Card error: {str(e)}")
            success = False

        mark_alert_sent(run_id, success)

        if success:
            alerts_sent += 1
            print(f"Alert sent ✅")
        else:
            alerts_failed += 1
            print(f"Alert failed ❌")

        print()

    print(f"Dispatch complete.")
    print(f"  Sent   : {alerts_sent}")
    print(f"  Failed : {alerts_failed}")

## Cell 7: Verify Alert Status

In [0]:
print("Alert status:\n")
spark.sql(f"""
    SELECT
        business_date,
        SUBSTRING_INDEX(
            source_fully_qualified_name, '.', -1
        )                               AS table_name,
        overall_status,
        ai_enrichment_status,
        CASE WHEN alert_sent = true
             THEN 'YES' ELSE 'NO'
        END                             AS alert_fired,
        alert_sent_timestamp,
        alert_channel
    FROM {RECONCILIATION_RUN}
    ORDER BY business_date, source_fully_qualified_name
""").show(20, truncate=False)

## Cell 8: Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 2 ALERT DISPATCHER COMPLETE")
print("=" * 60)

if NO_RUNS_TO_ALERT:
    print("Alerts sent   : 0 (no runs required alerting)")
else:
    print(f"Alerts sent   : {alerts_sent}")
    print(f"Alerts failed : {alerts_failed}")
    print(f"Channel       : {ALERT_CHANNEL}")

print(f"Completed at  : {datetime.now()}")
print("=" * 60)

In [0]:
%sql
-- Final verification of all runs and alerts

SELECT
    business_date,
    SUBSTRING_INDEX(
        source_fully_qualified_name, '.', -1
    )                                               AS table_name,
    overall_status,
    total_checks_passed,
    total_checks_warned,
    total_checks_critical,
    ai_enrichment_status,
    CASE WHEN alert_sent = true THEN 'YES' ELSE 'NO' END AS alert_fired,
    alert_channel
FROM learning_dev.s819534.uc2_reconciliation_run_b
ORDER BY business_date, source_fully_qualified_name;